# 14 integrate copykat

**Purpose:** Integrate completed CopyKAT microbatch predictions.

**Original analysis notebook:** `14_integrate_CopyKAT_microbatch_predictions_HVG8000_r0_6.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 33 — Integrate CopyKAT microbatch predictions

This notebook merges the 176 CopyKAT microbatch outputs into one final per-cell CopyKAT call.

Important design:
- Each microbatch contains **target cells** plus repeated **reference anchor cells**.
- Final whole-project calls are made from **target-cell predictions only**.
- Reference-anchor predictions are summarized separately for QC, but are not allowed to overweight the final result.
- All outputs are written with replace semantics.


In [ ]:

# =========================
# SETTINGS
# =========================
from pathlib import Path
import os, re, tarfile, tempfile, shutil, warnings
import numpy as np
import pandas as pd
import scanpy as sc

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
METADATA_DIR = PROJECT_DIR / "metadata"
PROCESSED_DIR = PROJECT_DIR / "processed/integrated"
COPYKAT_MICRO_DIR = PROJECT_DIR / "data/interim/integrated/CNV_inputs/CopyKAT_HVG8000_r0_6/batchwise_micro_all_cells"

INPUT_H5AD_CANDIDATES = [
    PROCESSED_DIR / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status.h5ad",
    PROCESSED_DIR / "GBM_core_4datasets_fullgene_marker_based_annotation.h5ad",
    PROCESSED_DIR / "GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.h5ad",
]

STATUS_TSV = METADATA_DIR / "copykat_microbatch_RStudio_simple_status_HVG8000_r0_6.tsv"
BATCH_PLAN_TSV = METADATA_DIR / "copykat_microbatch_batch_plan_HVG8000_r0_6.tsv"
INPUT_MANIFEST_TSV = METADATA_DIR / "copykat_microbatch_input_manifest_HVG8000_r0_6.tsv"

# Optional fallback if local prediction files are unavailable:
PREDICTION_BUNDLE_TAR_GZ = METADATA_DIR / "copykat_microbatch_predictions_bundle_HVG8000_r0_6.tar.gz"

OUT_ALL_ROWS_TSV = METADATA_DIR / "copykat_microbatch_all_prediction_rows_HVG8000_r0_6.tsv"
OUT_TARGET_ONLY_TSV = METADATA_DIR / "copykat_microbatch_target_only_predictions_HVG8000_r0_6.tsv"
OUT_FINAL_CELL_CALLS_TSV = METADATA_DIR / "copykat_microbatch_final_cell_calls_HVG8000_r0_6.tsv"
OUT_OVERALL_SUMMARY_TSV = METADATA_DIR / "copykat_microbatch_overall_summary_HVG8000_r0_6.tsv"
OUT_BATCH_QC_TSV = METADATA_DIR / "copykat_microbatch_batch_qc_summary_HVG8000_r0_6.tsv"
OUT_CLUSTER_SUMMARY_TSV = METADATA_DIR / "copykat_microbatch_cluster_summary_HVG8000_r0_6.tsv"
OUT_ANNOTATION_SUMMARY_TSV = METADATA_DIR / "copykat_microbatch_annotation_summary_HVG8000_r0_6.tsv"
OUT_DATASET_SUMMARY_TSV = METADATA_DIR / "copykat_microbatch_dataset_summary_HVG8000_r0_6.tsv"
OUT_CONDITION_SUMMARY_TSV = METADATA_DIR / "copykat_microbatch_condition_summary_HVG8000_r0_6.tsv"
OUT_PATIENT_SUMMARY_TSV = METADATA_DIR / "copykat_microbatch_patient_summary_HVG8000_r0_6.tsv"
OUT_CLUSTER_CONDITION_TSV = METADATA_DIR / "copykat_microbatch_cluster_condition_summary_HVG8000_r0_6.tsv"
OUT_CLUSTER_INTERPRETATION_TSV = METADATA_DIR / "copykat_microbatch_cluster_interpretation_HVG8000_r0_6.tsv"
OUT_OUTPUT_MANIFEST_TSV = METADATA_DIR / "copykat_microbatch_integration_output_manifest_HVG8000_r0_6.tsv"

OUT_H5AD = PROCESSED_DIR / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_microbatch.h5ad"

CLUSTER_KEY_CANDIDATES = ["leiden_scvi_main", "leiden_scvi_HVG8000_r0_6", "leiden_scvi_0.6", "leiden"]
ANNOTATION_COL_CANDIDATES = [
    "annotation_marker_based_HVG8000_r0_6",
    "annotation_marker_based",
    "cell_type_marker_based",
    "annotation",
]
DATASET_COL_CANDIDATES = ["core_dataset", "dataset_id", "dataset"]
CONDITION_COL_CANDIDATES = ["condition", "disease", "diagnosis"]
PATIENT_COL_CANDIDATES = ["patient_id", "patient", "Patient"]
SAMPLE_COL_CANDIDATES = ["sample_id", "sample", "Sample"]

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError("None of these paths exists:\n" + "\n".join(map(str, paths)))

def choose_col(df, candidates, required=False, label="column"):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"No {label} found. Tried: {candidates}. Available: {list(df.columns)[:80]}")
    return None

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")

def write_h5ad_replace(adata_obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    adata_obj.write_h5ad(path)
    print(f"[SAVED] {path}")

print("CopyKAT microbatch directory:", COPYKAT_MICRO_DIR)


In [ ]:

# =========================
# READ MAIN H5AD
# =========================
INPUT_H5AD = first_existing(INPUT_H5AD_CANDIDATES)
print("[READ H5AD]", INPUT_H5AD)
adata = sc.read_h5ad(INPUT_H5AD)
print(adata)

obs = adata.obs.copy()
obs["cell_id"] = obs.index.astype(str)

CLUSTER_KEY = choose_col(obs, CLUSTER_KEY_CANDIDATES, required=True, label="cluster key")
ANNOTATION_COL = choose_col(obs, ANNOTATION_COL_CANDIDATES, required=False)
DATASET_COL = choose_col(obs, DATASET_COL_CANDIDATES, required=False)
CONDITION_COL = choose_col(obs, CONDITION_COL_CANDIDATES, required=False)
PATIENT_COL = choose_col(obs, PATIENT_COL_CANDIDATES, required=False)
SAMPLE_COL = choose_col(obs, SAMPLE_COL_CANDIDATES, required=False)

print("CLUSTER_KEY:", CLUSTER_KEY)
print("ANNOTATION_COL:", ANNOTATION_COL)
print("DATASET_COL:", DATASET_COL)
print("CONDITION_COL:", CONDITION_COL)
print("PATIENT_COL:", PATIENT_COL)
print("SAMPLE_COL:", SAMPLE_COL)


In [ ]:

# =========================
# READ STATUS / PLAN / MANIFEST
# =========================
status_df = pd.read_csv(STATUS_TSV, sep="\t") if STATUS_TSV.exists() else pd.DataFrame()
batch_plan = pd.read_csv(BATCH_PLAN_TSV, sep="\t") if BATCH_PLAN_TSV.exists() else pd.DataFrame()
input_manifest = pd.read_csv(INPUT_MANIFEST_TSV, sep="\t") if INPUT_MANIFEST_TSV.exists() else pd.DataFrame()

print("status:", status_df.shape)
print("batch_plan:", batch_plan.shape)
print("input_manifest:", input_manifest.shape)

if len(status_df):
    display(status_df["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="n"))
if len(batch_plan):
    print("batch_plan target cells sum:", batch_plan.get("n_target_cells", pd.Series(dtype=int)).sum())
if len(input_manifest):
    print("input_manifest exported cells sum:", input_manifest.get("n_cells_exported", pd.Series(dtype=int)).sum())


In [ ]:

# =========================
# FIND LOCAL PREDICTION FILES
# =========================
pred_files = sorted(COPYKAT_MICRO_DIR.glob("copykat_microbatch_*/copykat_microbatch_*_prediction.tsv"))
print("local prediction files:", len(pred_files))

if len(pred_files) == 0 and PREDICTION_BUNDLE_TAR_GZ.exists():
    print("[FALLBACK] extracting prediction bundle:", PREDICTION_BUNDLE_TAR_GZ)
    tmp_dir = Path(tempfile.mkdtemp(prefix="copykat_microbatch_bundle_"))
    with tarfile.open(PREDICTION_BUNDLE_TAR_GZ, "r:gz") as tf:
        tf.extractall(tmp_dir)
    pred_files = sorted(tmp_dir.glob("copykat_microbatch_*/copykat_microbatch_*_prediction.tsv"))
    print("bundle prediction files:", len(pred_files))
    # Metadata files are not in the compact bundle. If local metadata is missing,
    # final calls will use all rows with duplicate voting. Local metadata is strongly preferred.

if len(pred_files) == 0:
    raise FileNotFoundError("No microbatch prediction TSV files found.")

pred_files[:3], pred_files[-3:]


In [ ]:

# =========================
# READ ALL PREDICTIONS AND LOCAL METADATA
# =========================
rows = []
batch_qc_rows = []
missing_meta_batches = []

for pred_path in pred_files:
    batch_dir = pred_path.parent
    batch_id_match = re.search(r"copykat_microbatch_\d{4}", str(pred_path))
    batch_id = batch_id_match.group(0) if batch_id_match else batch_dir.name

    pred = pd.read_csv(pred_path, sep="\t")
    pred.columns = [str(c).strip() for c in pred.columns]
    if "cell.names" not in pred.columns:
        # tolerate alternate exported names
        cell_col = choose_col(pred, ["cell_id", "cell", "barcode", "Cell"], required=True, label="prediction cell id")
        pred = pred.rename(columns={cell_col: "cell.names"})
    if "copykat.pred" not in pred.columns:
        pred_col = choose_col(pred, ["copykat_pred", "prediction", "pred", "copykat.prediction"], required=True, label="prediction call")
        pred = pred.rename(columns={pred_col: "copykat.pred"})

    pred = pred[["cell.names", "copykat.pred"]].copy()
    pred["cell_id"] = pred["cell.names"].astype(str)
    pred["copykat_pred_raw"] = pred["copykat.pred"].astype(str)
    pred["batch_id"] = batch_id
    pred["prediction_file"] = str(pred_path)

    meta_path = batch_dir / "copykat_cell_metadata.tsv"
    if meta_path.exists():
        meta = pd.read_csv(meta_path, sep="\t")
        meta.columns = [str(c).strip() for c in meta.columns]
        if "cell_id" in meta.columns:
            keep_cols = ["cell_id"]
            for c in [
                "original_index", "is_target_cell", "is_reference_anchor",
                "batch_id", CLUSTER_KEY, ANNOTATION_COL, DATASET_COL,
                CONDITION_COL, PATIENT_COL, SAMPLE_COL, "CNV_role_recommendation"
            ]:
                if c is not None and c in meta.columns and c not in keep_cols:
                    keep_cols.append(c)
            meta = meta[keep_cols].copy()
            # Avoid batch_id collision.
            if "batch_id" in meta.columns:
                meta = meta.rename(columns={"batch_id": "metadata_batch_id"})
            pred = pred.merge(meta, on="cell_id", how="left")
        else:
            missing_meta_batches.append(batch_id)
            pred["is_target_cell"] = np.nan
            pred["is_reference_anchor"] = np.nan
    else:
        missing_meta_batches.append(batch_id)
        pred["is_target_cell"] = np.nan
        pred["is_reference_anchor"] = np.nan

    rows.append(pred)
    batch_qc_rows.append({
        "batch_id": batch_id,
        "prediction_file": str(pred_path),
        "metadata_file_exists": bool(meta_path.exists()),
        "n_prediction_rows": int(len(pred)),
        "n_aneuploid_rows": int((pred["copykat_pred_raw"] == "aneuploid").sum()),
        "n_diploid_rows": int((pred["copykat_pred_raw"] == "diploid").sum()),
        "n_not_defined_rows": int((pred["copykat_pred_raw"] == "not.defined").sum()),
    })

all_pred = pd.concat(rows, ignore_index=True)
batch_qc = pd.DataFrame(batch_qc_rows)

# Normalize boolean metadata.
for c in ["is_target_cell", "is_reference_anchor"]:
    if c in all_pred.columns:
        all_pred[c] = all_pred[c].map(lambda x: True if str(x).lower() == "true" else (False if str(x).lower() == "false" else np.nan))

print("all prediction rows:", all_pred.shape)
print("unique cells represented:", all_pred["cell_id"].nunique())
print("missing metadata batches:", len(set(missing_meta_batches)))
display(all_pred["copykat_pred_raw"].value_counts(dropna=False).rename_axis("copykat_pred_raw").reset_index(name="n"))
display(batch_qc.head())
write_tsv_replace(batch_qc, OUT_BATCH_QC_TSV)


In [ ]:

# =========================
# TARGET-ONLY FINAL CALLS
# =========================
has_target_metadata = "is_target_cell" in all_pred.columns and all_pred["is_target_cell"].notna().any()

if has_target_metadata:
    target_pred = all_pred[all_pred["is_target_cell"] == True].copy()
    anchor_pred = all_pred[all_pred["is_reference_anchor"] == True].copy()
    print("target prediction rows:", target_pred.shape)
    print("anchor/reference prediction rows:", anchor_pred.shape)
else:
    warnings.warn("No target metadata found. Falling back to duplicate-vote final calls over all prediction rows.")
    target_pred = all_pred.copy()
    anchor_pred = all_pred.iloc[0:0].copy()

write_tsv_replace(all_pred, OUT_ALL_ROWS_TSV)
write_tsv_replace(target_pred, OUT_TARGET_ONLY_TSV)

# For target cells, there should be one prediction per true target cell.
target_dup = target_pred.groupby("cell_id").size().rename("n_target_prediction_rows").reset_index()
display(target_dup["n_target_prediction_rows"].describe())

def majority_call(vals):
    vc = pd.Series(vals).value_counts(dropna=False)
    if len(vc) == 0:
        return "missing"
    if len(vc) == 1:
        return str(vc.index[0])
    if vc.iloc[0] > vc.iloc[1]:
        return str(vc.index[0])
    # If tie happens, keep it explicit rather than forcing a biologically misleading call.
    winners = sorted([str(i) for i, v in vc.items() if v == vc.iloc[0]])
    return "tie__" + "__".join(winners)

agg = (
    target_pred
    .groupby("cell_id", as_index=False)
    .agg(
        copykat_microbatch_pred=("copykat_pred_raw", majority_call),
        copykat_microbatch_n_rows=("copykat_pred_raw", "size"),
        copykat_microbatch_batches=("batch_id", lambda x: ";".join(sorted(set(map(str, x))))),
    )
)

# Attach target metadata, one row per cell.
meta_cols = ["cell_id"]
for c in ["original_index", CLUSTER_KEY, ANNOTATION_COL, DATASET_COL, CONDITION_COL, PATIENT_COL, SAMPLE_COL, "CNV_role_recommendation"]:
    if c is not None and c in target_pred.columns and c not in meta_cols:
        meta_cols.append(c)
target_meta = target_pred[meta_cols].drop_duplicates("cell_id")
final_calls = agg.merge(target_meta, on="cell_id", how="left")

# Add cells missing from CopyKAT target output.
all_cells = pd.DataFrame({"cell_id": adata.obs_names.astype(str)})
final_calls = all_cells.merge(final_calls, on="cell_id", how="left")
final_calls["copykat_microbatch_pred"] = final_calls["copykat_microbatch_pred"].fillna("missing")
final_calls["copykat_microbatch_defined"] = final_calls["copykat_microbatch_pred"].isin(["aneuploid", "diploid"])
final_calls["copykat_microbatch_is_aneuploid"] = final_calls["copykat_microbatch_pred"].eq("aneuploid")
final_calls["copykat_microbatch_is_diploid"] = final_calls["copykat_microbatch_pred"].eq("diploid")
final_calls["copykat_microbatch_is_not_defined"] = final_calls["copykat_microbatch_pred"].eq("not.defined")
final_calls["copykat_microbatch_is_missing"] = final_calls["copykat_microbatch_pred"].eq("missing")

print("final cell calls:", final_calls.shape)
display(final_calls["copykat_microbatch_pred"].value_counts(dropna=False).rename_axis("copykat_microbatch_pred").reset_index(name="n"))
write_tsv_replace(final_calls, OUT_FINAL_CELL_CALLS_TSV)


In [ ]:

# =========================
# OVERALL SUMMARY
# =========================
overall = pd.DataFrame([{
    "input_h5ad": str(INPUT_H5AD),
    "copykat_microbatch_dir": str(COPYKAT_MICRO_DIR),
    "n_adata_cells": int(adata.n_obs),
    "n_prediction_files": int(len(pred_files)),
    "n_prediction_rows_total_including_anchors": int(len(all_pred)),
    "n_unique_cells_in_all_prediction_rows": int(all_pred["cell_id"].nunique()),
    "n_target_prediction_rows": int(len(target_pred)),
    "n_final_cells": int(len(final_calls)),
    "n_final_aneuploid": int(final_calls["copykat_microbatch_pred"].eq("aneuploid").sum()),
    "n_final_diploid": int(final_calls["copykat_microbatch_pred"].eq("diploid").sum()),
    "n_final_not_defined": int(final_calls["copykat_microbatch_pred"].eq("not.defined").sum()),
    "n_final_missing": int(final_calls["copykat_microbatch_pred"].eq("missing").sum()),
    "aneuploid_fraction_all_cells": float(final_calls["copykat_microbatch_pred"].eq("aneuploid").mean()),
    "aneuploid_fraction_defined_cells": float(
        final_calls["copykat_microbatch_pred"].eq("aneuploid").sum()
        / max(1, final_calls["copykat_microbatch_pred"].isin(["aneuploid", "diploid"]).sum())
    ),
    "note": "Final calls are target-cell predictions only; repeated reference-anchor rows are not used for final per-cell calls.",
}])
write_tsv_replace(overall, OUT_OVERALL_SUMMARY_TSV)
display(overall.T)


In [ ]:

# =========================
# SUMMARIZATION HELPERS
# =========================
def summarize_by(df, group_cols):
    group_cols = [c for c in group_cols if c is not None and c in df.columns]
    if len(group_cols) == 0:
        return pd.DataFrame()
    tmp = df.copy()
    for c in group_cols:
        tmp[c] = tmp[c].astype(str).fillna("NA")

    out = (
        tmp.groupby(group_cols, observed=True)
        .agg(
            n_cells=("cell_id", "count"),
            n_aneuploid=("copykat_microbatch_is_aneuploid", "sum"),
            n_diploid=("copykat_microbatch_is_diploid", "sum"),
            n_not_defined=("copykat_microbatch_is_not_defined", "sum"),
            n_missing=("copykat_microbatch_is_missing", "sum"),
        )
        .reset_index()
    )
    out["n_defined"] = out["n_aneuploid"] + out["n_diploid"]
    out["aneuploid_fraction_all"] = out["n_aneuploid"] / out["n_cells"].replace(0, np.nan)
    out["aneuploid_fraction_defined"] = out["n_aneuploid"] / out["n_defined"].replace(0, np.nan)
    out["not_defined_fraction"] = out["n_not_defined"] / out["n_cells"].replace(0, np.nan)
    out["missing_fraction"] = out["n_missing"] / out["n_cells"].replace(0, np.nan)
    return out.sort_values(["aneuploid_fraction_defined", "n_cells"], ascending=[False, False])

cluster_summary = summarize_by(final_calls, [CLUSTER_KEY])
annotation_summary = summarize_by(final_calls, [ANNOTATION_COL])
dataset_summary = summarize_by(final_calls, [DATASET_COL])
condition_summary = summarize_by(final_calls, [CONDITION_COL])
patient_summary = summarize_by(final_calls, [PATIENT_COL])
cluster_condition_summary = summarize_by(final_calls, [CLUSTER_KEY, CONDITION_COL])

write_tsv_replace(cluster_summary, OUT_CLUSTER_SUMMARY_TSV)
if len(annotation_summary): write_tsv_replace(annotation_summary, OUT_ANNOTATION_SUMMARY_TSV)
if len(dataset_summary): write_tsv_replace(dataset_summary, OUT_DATASET_SUMMARY_TSV)
if len(condition_summary): write_tsv_replace(condition_summary, OUT_CONDITION_SUMMARY_TSV)
if len(patient_summary): write_tsv_replace(patient_summary, OUT_PATIENT_SUMMARY_TSV)
if len(cluster_condition_summary): write_tsv_replace(cluster_condition_summary, OUT_CLUSTER_CONDITION_TSV)

display(cluster_summary)


In [ ]:

# =========================
# CLUSTER-LEVEL INTERPRETATION
# =========================
interpret = cluster_summary.copy()

if ANNOTATION_COL is not None and ANNOTATION_COL in final_calls.columns:
    ann_map = (
        final_calls[[CLUSTER_KEY, ANNOTATION_COL]]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .groupby(CLUSTER_KEY)[ANNOTATION_COL]
        .apply(lambda x: ";".join(sorted(set(x))))
        .reset_index()
    )
    interpret = interpret.merge(ann_map, on=CLUSTER_KEY, how="left")

def cluster_call(row):
    n_defined = row.get("n_defined", 0)
    frac = row.get("aneuploid_fraction_defined", np.nan)
    if pd.isna(frac) or n_defined < 30:
        return "insufficient_defined_cells"
    if frac >= 0.65:
        return "copykat_aneuploid_high"
    if frac >= 0.40:
        return "copykat_aneuploid_mixed_moderate_high"
    if frac >= 0.20:
        return "copykat_aneuploid_low_mixed"
    return "copykat_diploid_dominant"

interpret["copykat_cluster_call"] = interpret.apply(cluster_call, axis=1)

def biological_interpretation(row):
    ann = str(row.get(ANNOTATION_COL, "")).lower() if ANNOTATION_COL is not None else ""
    call = row["copykat_cluster_call"]
    frac = row.get("aneuploid_fraction_defined", np.nan)
    if "tumor" in ann and call in ["copykat_aneuploid_high", "copykat_aneuploid_mixed_moderate_high"]:
        return "marker_tumor_like_with_copykat_support"
    if "tumor" in ann and call == "copykat_aneuploid_low_mixed":
        return "marker_tumor_like_with_weak_or_mixed_copykat_support"
    if any(x in ann for x in ["endothelial", "oligodendrocyte", "myeloid", "t_nk", "b_plasma"]) and frac < 0.20:
        return "non_malignant_reference_like_by_copykat"
    if call == "copykat_aneuploid_high":
        return "copykat_high_aneuploid_review_annotation"
    return "review_with_marker_and_composition_context"

interpret["copykat_biological_interpretation"] = interpret.apply(biological_interpretation, axis=1)
interpret = interpret.sort_values(["aneuploid_fraction_defined", "n_cells"], ascending=[False, False])
write_tsv_replace(interpret, OUT_CLUSTER_INTERPRETATION_TSV)
display(interpret)


In [ ]:

# =========================
# ADD COPYKAT CALLS TO H5AD AND SAVE
# =========================
fc = final_calls.set_index("cell_id").reindex(adata.obs_names.astype(str))

for col in [
    "copykat_microbatch_pred",
    "copykat_microbatch_n_rows",
    "copykat_microbatch_batches",
    "copykat_microbatch_defined",
    "copykat_microbatch_is_aneuploid",
    "copykat_microbatch_is_diploid",
    "copykat_microbatch_is_not_defined",
    "copykat_microbatch_is_missing",
]:
    if col in fc.columns:
        adata.obs[col] = fc[col].values

# Categorical cleanup
for col in ["copykat_microbatch_pred", "copykat_microbatch_batches"]:
    if col in adata.obs.columns:
        adata.obs[col] = adata.obs[col].astype("category")

write_h5ad_replace(adata, OUT_H5AD)
print(adata)


In [ ]:

# =========================
# OUTPUT MANIFEST
# =========================
manifest_rows = []
for label, path in [
    ("all_prediction_rows", OUT_ALL_ROWS_TSV),
    ("target_only_predictions", OUT_TARGET_ONLY_TSV),
    ("final_cell_calls", OUT_FINAL_CELL_CALLS_TSV),
    ("overall_summary", OUT_OVERALL_SUMMARY_TSV),
    ("batch_qc_summary", OUT_BATCH_QC_TSV),
    ("cluster_summary", OUT_CLUSTER_SUMMARY_TSV),
    ("annotation_summary", OUT_ANNOTATION_SUMMARY_TSV),
    ("dataset_summary", OUT_DATASET_SUMMARY_TSV),
    ("condition_summary", OUT_CONDITION_SUMMARY_TSV),
    ("patient_summary", OUT_PATIENT_SUMMARY_TSV),
    ("cluster_condition_summary", OUT_CLUSTER_CONDITION_TSV),
    ("cluster_interpretation", OUT_CLUSTER_INTERPRETATION_TSV),
    ("annotated_h5ad_with_copykat", OUT_H5AD),
]:
    path = Path(path)
    manifest_rows.append({
        "output_label": label,
        "path": str(path),
        "exists": bool(path.exists()),
        "size_mb": round(path.stat().st_size / 1024**2, 3) if path.exists() else np.nan,
    })

out_manifest = pd.DataFrame(manifest_rows)
write_tsv_replace(out_manifest, OUT_OUTPUT_MANIFEST_TSV)
display(out_manifest)

print("\nSend these after running:")
for p in [
    OUT_OVERALL_SUMMARY_TSV,
    OUT_CLUSTER_SUMMARY_TSV,
    OUT_ANNOTATION_SUMMARY_TSV,
    OUT_DATASET_SUMMARY_TSV,
    OUT_CONDITION_SUMMARY_TSV,
    OUT_CLUSTER_INTERPRETATION_TSV,
    OUT_OUTPUT_MANIFEST_TSV,
]:
    print(p)
print("\nDo not send the h5ad unless specifically needed:", OUT_H5AD)
